## 📦 CELL 1 — Install Semua Dependencies

In [1]:
# ── HF MONKEYPATCH ──
import huggingface_hub
try:
    from huggingface_hub import cached_download
except ImportError:
    import huggingface_hub.file_download
    huggingface_hub.cached_download = huggingface_hub.hf_hub_download
    huggingface_hub.file_download.cached_download = huggingface_hub.hf_hub_download
try:
    from huggingface_hub.utils import HfFolder
except ImportError:
    try:
        from huggingface_hub import HfFolder
    except ImportError:
        class HfFolder:
            @classmethod
            def get_token(cls): import os; return os.environ.get('HF_TOKEN')
            @classmethod
            def save_token(cls, token): pass
            @classmethod
            def delete_token(cls): pass
    import huggingface_hub.utils
    huggingface_hub.utils.HfFolder = HfFolder
    huggingface_hub.HfFolder = HfFolder
import huggingface_hub.constants
if not hasattr(huggingface_hub.constants, 'HF_HUB_ENABLE_HF_TRANSFER'):
    huggingface_hub.constants.HF_HUB_ENABLE_HF_TRANSFER = False
# ───────────────────

import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-800:])
    else:
        print(result.stdout[-300:] or 'OK')

print('=== [1/5] Face Swap packages ===')
run('pip install -q insightface==0.7.3 onnxruntime-gpu==1.19.2 opencv-python-headless Pillow')
run('pip install -q gfpgan facexlib basicsr')

print('\n=== [2/5] AI Video Generator packages ===')
try:
    import diffusers, transformers, huggingface_hub
    print('✓ Core AI libraries (diffusers, transformers, huggingface_hub) are already installed. Skipping install to prevent memory mismatch.')
except ImportError:
    run('pip install -q --upgrade diffusers transformers accelerate huggingface_hub')
run('pip install -q xformers peft')
run('pip install -q vtracer scipy')

print('\n=== [3/5] Server & tunnel packages ===')
run('pip install -q fastapi uvicorn[standard] python-multipart nest_asyncio pyngrok tqdm pydantic requests')

print('\n=== [4/5] ffmpeg ===')
run('apt-get install -y ffmpeg -qq')

print('\n=== [5/5] Verifikasi GPU ===')
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "⚠️ GPU TIDAK AKTIF!"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')
print('\n✅ Semua dependencies berhasil diinstall!')

=== [1/5] Face Swap packages ===
━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.2/226.2 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.0 MB/s eta 0:00:00

━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.3/338.3 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 17.2 MB/s eta 0:00:00


=== [2/5] AI Video Generator packages ===
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.0/419.0 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.5 MB/s eta 0:00:00

   ━━━━━━━━━━━━━━━━━━

# 🧠 VAX MODEL — Kaggle AI Engine
**Unified pipeline: Face Swap + AI Video Generator**

| Fitur | Model |
|---|---|
| 🎭 Face Swap Foto | InsightFace + inswapper_128 + GFPGAN |
| 🎬 Face Swap Video | InsightFace + inswapper_128 + GFPGAN |
| 🖼️ Text → Image | Stable Diffusion v1.5 |
| 📹 Image → Video | Stable Video Diffusion XT |
| 🧠 Image → Video HD | CogVideoX-5B |

**Alur:**
```
[Kaggle GPU T4] ──ngrok──▶ [Local UI]
     ↑                            ↓
  GPU heavy work          Kirim request & download hasil
```
> ⚠️ Jalankan setiap cell secara berurutan. Pastikan GPU T4 aktif di: Settings → Accelerator → GPU T4 x2

## 📁 CELL 2 — Setup Direktori & Download Model Face Swap

In [2]:
import os
import urllib.request
from pathlib import Path
from tqdm import tqdm
import torch

# ── Direktori Kaggle ──────────────────────────────────────────────
WORK_DIR   = Path('/kaggle/working')
MODEL_DIR  = WORK_DIR / 'models'
INPUT_DIR  = WORK_DIR / 'input'
OUTPUT_DIR = WORK_DIR / 'output'
TEMP_DIR   = WORK_DIR / 'temp'
UPLOAD_DIR = WORK_DIR / 'uploads'
GEN_DIR    = WORK_DIR / 'generated'   # hasil AI generate

for d in [MODEL_DIR, INPUT_DIR, OUTPUT_DIR, TEMP_DIR, UPLOAD_DIR, GEN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('📁 Struktur folder dibuat:')
for d in [MODEL_DIR, INPUT_DIR, OUTPUT_DIR, TEMP_DIR, UPLOAD_DIR, GEN_DIR]:
    print(f'   {d}')

# ── Cache HuggingFace ke /kaggle/working ──────────────────────────
HF_CACHE = str(MODEL_DIR / 'hf_cache')
os.environ['HF_HOME']                   = HF_CACHE
os.environ['PYTORCH_CUDA_ALLOC_CONF']   = 'max_split_size_mb:128,expandable_segments:True'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL']      = '3'
os.makedirs(HF_CACHE, exist_ok=True)

# ── Helper download ───────────────────────────────────────────────
opener = urllib.request.build_opener()
opener.addheaders = [('User-agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)')]
urllib.request.install_opener(opener)

class DownloadProgress(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize: self.total = tsize
        self.update(b * bsize - self.n)

# ── InsightFace buffalo_l ─────────────────────────────────────────
import insightface
from insightface.app import FaceAnalysis

print('\n📥 Downloading InsightFace buffalo_l...')
_face_app_init = FaceAnalysis(
    name='buffalo_l',
    root=str(MODEL_DIR),
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)
_face_app_init.prepare(ctx_id=0, det_size=(640, 640))
del _face_app_init
print('✅ InsightFace buffalo_l siap!')

# ── inswapper_128.onnx ────────────────────────────────────────────
SWAP_MODEL_PATH = MODEL_DIR / 'inswapper_128.onnx'
if not SWAP_MODEL_PATH.exists():
    print('\n📥 Downloading inswapper_128.onnx (~256MB)...')
    url = 'https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx'
    with DownloadProgress(unit='B', unit_scale=True, miniters=1, desc='inswapper_128.onnx') as t:
        urllib.request.urlretrieve(url, SWAP_MODEL_PATH, reporthook=t.update_to)
    print(f'✅ Saved: {SWAP_MODEL_PATH}')
else:
    print(f'✅ inswapper_128.onnx sudah ada')

# ── GFPGANv1.4.pth ───────────────────────────────────────────────
GFPGAN_MODEL_PATH = MODEL_DIR / 'GFPGANv1.4.pth'
if not GFPGAN_MODEL_PATH.exists():
    print('\n📥 Downloading GFPGANv1.4.pth (~330MB)...')
    url = 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth'
    with DownloadProgress(unit='B', unit_scale=True, miniters=1, desc='GFPGANv1.4.pth') as t:
        urllib.request.urlretrieve(url, GFPGAN_MODEL_PATH, reporthook=t.update_to)
    print(f'✅ Saved: {GFPGAN_MODEL_PATH}')
else:
    print(f'✅ GFPGANv1.4.pth sudah ada')

print('\n🎉 Semua model face swap siap! Model AI Video akan di-download saat pertama dipakai.')

📁 Struktur folder dibuat:
   /kaggle/working/models
   /kaggle/working/input
   /kaggle/working/output
   /kaggle/working/temp
   /kaggle/working/uploads
   /kaggle/working/generated

📥 Downloading InsightFace buffalo_l...
download_path: /kaggle/working/models/models/buffalo_l


100%|██████████| 281857/281857 [00:02<00:00, 112106.33KB/s]


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /kaggle/working/models/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecut

inswapper_128.onnx: 554MB [00:02, 276MB/s]                               


✅ Saved: /kaggle/working/models/inswapper_128.onnx

📥 Downloading GFPGANv1.4.pth (~330MB)...


GFPGANv1.4.pth: 349MB [00:03, 99.6MB/s]                              

✅ Saved: /kaggle/working/models/GFPGANv1.4.pth

🎉 Semua model face swap siap! Model AI Video akan di-download saat pertama dipakai.


## ⚙️ CELL 3 — Load Face Swap Models & Core Functions

In [3]:
# ── HF MONKEYPATCH ──
import huggingface_hub
try:
    from huggingface_hub import cached_download
except ImportError:
    import huggingface_hub.file_download
    huggingface_hub.cached_download = huggingface_hub.hf_hub_download
    huggingface_hub.file_download.cached_download = huggingface_hub.hf_hub_download
try:
    from huggingface_hub.utils import HfFolder
except ImportError:
    try:
        from huggingface_hub import HfFolder
    except ImportError:
        class HfFolder:
            @classmethod
            def get_token(cls): import os; return os.environ.get('HF_TOKEN')
            @classmethod
            def save_token(cls, token): pass
            @classmethod
            def delete_token(cls): pass
    import huggingface_hub.utils
    huggingface_hub.utils.HfFolder = HfFolder
    huggingface_hub.HfFolder = HfFolder
import huggingface_hub.constants
if not hasattr(huggingface_hub.constants, 'HF_HUB_ENABLE_HF_TRANSFER'):
    huggingface_hub.constants.HF_HUB_ENABLE_HF_TRANSFER = False
# ───────────────────

import cv2
import numpy as np
import insightface
from insightface.app import FaceAnalysis
from pathlib import Path
from PIL import Image
import torch
import sys
import torchvision

WORK_DIR   = Path('/kaggle/working')
MODEL_DIR  = WORK_DIR / 'models'
OUTPUT_DIR = WORK_DIR / 'output'
TEMP_DIR   = WORK_DIR / 'temp'

# ── Load Face Analyzer ───────────────────────────────────────────
print('🔄 Loading InsightFace FaceAnalysis...')
face_app = FaceAnalysis(
    name='buffalo_l',
    root=str(MODEL_DIR),
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)
face_app.prepare(ctx_id=0, det_size=(640, 640))
print('✅ FaceAnalysis loaded')

# ── Load Face Swapper ─────────────────────────────────────────────
print('🔄 Loading inswapper_128...')
face_swapper = insightface.model_zoo.get_model(
    str(MODEL_DIR / 'inswapper_128.onnx'),
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)
print('✅ Face swapper loaded')

# ── Load GFPGAN Enhancer ──────────────────────────────────────────
print('🔄 Loading GFPGAN...')
# Fix bug torchvision di basicsr
sys.modules['torchvision.transforms.functional_tensor'] = torchvision.transforms.functional

from gfpgan import GFPGANer

face_enhancer = GFPGANer(
    model_path=str(MODEL_DIR / 'GFPGANv1.4.pth'),
    upscale=1, arch='clean', channel_multiplier=2, bg_upsampler=None
)
print('✅ GFPGAN loaded')


# ═══════════════════════════════════════════════════════════════════
#  FACE SWAP CORE FUNCTIONS
# ═══════════════════════════════════════════════════════════════════

def get_face_embedding(img_bgr):
    """Ambil face embedding terbesar dari gambar BGR."""
    faces = face_app.get(img_bgr)
    if not faces:
        return None
    return sorted(faces, key=lambda f: f.bbox[2] - f.bbox[0], reverse=True)[0]


def swap_faces_in_frame(frame_bgr, source_face):
    """Swap semua wajah di frame dengan source_face."""
    target_faces = face_app.get(frame_bgr)
    if not target_faces:
        return frame_bgr
    result = frame_bgr.copy()
    for tf in target_faces:
        result = face_swapper.get(result, tf, source_face, paste_back=True)
    return result


def enhance_frame(frame_bgr):
    """Enhance wajah dengan GFPGAN."""
    _, _, enhanced = face_enhancer.enhance(
        frame_bgr, has_aligned=False, only_center_face=False, paste_back=True
    )
    return enhanced if enhanced is not None else frame_bgr


def process_image(source_path: str, target_path: str,
                  output_path: str, enhance: bool = True) -> str:
    """Face swap untuk satu gambar."""
    source_img = cv2.imread(source_path)
    target_img = cv2.imread(target_path)
    if source_img is None: raise ValueError(f'Gagal baca source: {source_path}')
    if target_img is None: raise ValueError(f'Gagal baca target: {target_path}')
    source_face = get_face_embedding(source_img)
    if source_face is None: raise ValueError('Tidak ada wajah terdeteksi di source image!')
    result = swap_faces_in_frame(target_img, source_face)
    if enhance:
        result = enhance_frame(result)
    cv2.imwrite(output_path, result)
    print(f'✅ Image saved: {output_path}')
    return output_path


def process_video(source_path: str, target_path: str,
                  output_path: str, enhance: bool = True,
                  fps_override: int = None) -> str:
    """Face swap untuk video frame-by-frame."""
    from tqdm import tqdm
    source_img = cv2.imread(source_path)
    if source_img is None: raise ValueError(f'Gagal baca source: {source_path}')
    source_face = get_face_embedding(source_img)
    if source_face is None: raise ValueError('Tidak ada wajah terdeteksi di source image!')

    cap = cv2.VideoCapture(target_path)
    if not cap.isOpened(): raise ValueError(f'Gagal buka video: {target_path}')

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps    = fps_override or cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f'📹 {total_frames} frames | {fps:.1f} fps | {width}x{height}')

    temp_video = str(TEMP_DIR / 'temp_swapped.mp4')
    writer = cv2.VideoWriter(temp_video, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    frame_idx = 0
    with tqdm(total=total_frames, desc='🎭 Swapping frames') as pbar:
        while True:
            ret, frame = cap.read()
            if not ret: break
            swapped = swap_faces_in_frame(frame, source_face)
            if enhance and frame_idx % 3 == 0:
                swapped = enhance_frame(swapped)
            writer.write(swapped)
            frame_idx += 1
            pbar.update(1)

    cap.release(); writer.release()

    print('🎵 Merging audio...')
    import subprocess
    cmd = (f'ffmpeg -y -i "{temp_video}" -i "{target_path}" '
           f'-c:v copy -c:a aac -map 0:v:0 -map 1:a:0 -shortest "{output_path}" 2>/dev/null')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        import shutil; shutil.copy(temp_video, output_path)
    print(f'✅ Video saved: {output_path}')
    return output_path


print('\n✅ Face Swap functions siap!')
print('   process_image(source, target, output, enhance=True)')
print('   process_video(source, target, output, enhance=True)')

🔄 Loading InsightFace FaceAnalysis...
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /kaggle/working/models/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutio

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Downloading: "https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth" to /kaggle/working/gfpgan/weights/detection_Resnet50_Final.pth



100%|██████████| 104M/104M [00:00<00:00, 241MB/s] 


Downloading: "https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth" to /kaggle/working/gfpgan/weights/parsing_parsenet.pth



100%|██████████| 81.4M/81.4M [00:00<00:00, 248MB/s]


✅ GFPGAN loaded

✅ Face Swap functions siap!
   process_image(source, target, output, enhance=True)
   process_video(source, target, output, enhance=True)


## 🧠 CELL 4 — AI Video Generator Functions (SD + SVD + CogVideoX)

In [4]:
import gc, time, base64, io, warnings, os
import numpy as np
import torch
from PIL import Image
from pathlib import Path

warnings.filterwarnings('ignore', category=FutureWarning)
torch.backends.cudnn.benchmark = True
if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

GEN_DIR = Path('/kaggle/working/generated')
GEN_DIR.mkdir(parents=True, exist_ok=True)

# ── Global state generator ────────────────────────────────────────
jobs            = {}       # Unified job tracker (dipakai oleh seluruh engine)
gen_pipe        = None
current_model_name = None
model_lock      = False


# ── Memory utils ──────────────────────────────────────────────────
def clear_memory(force: bool = False):
    global gen_pipe, current_model_name, model_lock
    if model_lock and not force:
        return
    print('\n🧹 Clearing GPU memory...')
    if gen_pipe is not None:
        del gen_pipe
    gen_pipe = None
    current_model_name = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.ipc_collect()
    time.sleep(0.5)


def get_free_memory():
    if torch.cuda.is_available():
        free  = torch.cuda.mem_get_info()[0] / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        return free, total
    return 0, 0


def check_memory_safe():
    free, _ = get_free_memory()
    if free < 1.5:
        print(f'⚠️ Low memory ({free:.1f} GB), cleaning...')
        clear_memory(force=True)


def _show_progress(job_id, step, total, model_name):
    pct = int((step / total) * 100)
    bar = '█' * (pct // 5) + '░' * (20 - pct // 5)
    print(f'\r🚀 [{model_name}] {job_id} |{bar}| {pct}%', end='', flush=True)
    if step % 5 == 0 and job_id in jobs:
        jobs[job_id]['progress'] = pct


# ── SD 1.5 — Text to Image ───────────────────────────────────────
def run_sd15(job_id: str, req: dict):
    global gen_pipe, current_model_name, model_lock
    try:
        model_lock = True
        check_memory_safe()

        if current_model_name != 'sd15':
            clear_memory()
            print('\n🖼️ Loading Stable Diffusion v1.5...')
            from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
            gen_pipe = StableDiffusionPipeline.from_pretrained(
                'runwayml/stable-diffusion-v1-5',
                torch_dtype=torch.float16, safety_checker=None, use_safetensors=True
            )
            gen_pipe.scheduler = DPMSolverMultistepScheduler.from_config(
                gen_pipe.scheduler.config, use_karras_sigmas=True
            )
            gen_pipe = gen_pipe.to('cuda')
            gen_pipe.enable_attention_slicing()
            gen_pipe.enable_vae_slicing()
            current_model_name = 'sd15'
            print(' ✓ SD 1.5 Ready')

        seed   = req.get('seed', -1)
        actual = seed if seed != -1 else int(np.random.randint(0, 1_000_000))
        gen    = torch.Generator(device='cuda').manual_seed(actual)
        steps  = min(req.get('num_inference_steps', 20), 25)

        with torch.inference_mode():
            image = gen_pipe(
                req.get('prompt', 'a beautiful landscape'),
                negative_prompt     = req.get('negative_prompt', 'blurry, low quality'),
                width               = req.get('width', 512),
                height              = req.get('height', 512),
                num_inference_steps = steps,
                guidance_scale      = req.get('guidance_scale', 7.5),
                generator           = gen,
                callback_on_step_end=lambda pipe, i, t, kw: (_show_progress(job_id, i, steps, 'SD1.5'), kw)[1]
            ).images[0]

        path = str(GEN_DIR / f'{job_id}.png')
        image.save(path, optimize=True)
        jobs[job_id].update({'status': 'done', 'file': path, 'progress': 100, 'seed': actual})
        print(f'\n✅ SD 1.5 selesai → {path}')

    except Exception as e:
        print(f'\n❌ SD1.5 error: {e}')
        jobs[job_id].update({'status': 'failed', 'error': str(e)[:300]})
        clear_memory(force=True)
    finally:
        model_lock = False


# ── SVD — Image to Video ─────────────────────────────────────────
def run_svd(job_id: str, req: dict):
    global gen_pipe, current_model_name, model_lock
    try:
        model_lock = True
        check_memory_safe()

        if current_model_name != 'svd':
            clear_memory()
            print('\n🎬 Loading Stable Video Diffusion XT (~10GB)...')
            from diffusers import StableVideoDiffusionPipeline
            gen_pipe = StableVideoDiffusionPipeline.from_pretrained(
                'stabilityai/stable-video-diffusion-img2vid-xt',
                torch_dtype=torch.float16, variant='fp16', use_safetensors=True
            )
            gen_pipe = gen_pipe.to('cuda')
            gen_pipe.enable_attention_slicing()
            gen_pipe.enable_sequential_cpu_offload()
            current_model_name = 'svd'
            print(' ✓ SVD Ready')

        img = Image.open(io.BytesIO(base64.b64decode(req.get('image_base64', '')))).convert('RGB')
        img = img.resize((req.get('width', 480), req.get('height', 272)))

        seed   = req.get('seed', -1)
        actual = seed if seed != -1 else int(np.random.randint(0, 1_000_000))
        gen    = torch.Generator(device='cuda').manual_seed(actual)
        n_frames = min(req.get('num_frames', 14), 14)

        with torch.inference_mode():
            frames = gen_pipe(
                img, decode_chunk_size=2,
                num_frames=n_frames, num_inference_steps=20,
                generator=gen,
                callback_on_step_end=lambda pipe, i, t, kw: (_show_progress(job_id, i, 20, 'SVD'), kw)[1]
            ).frames[0]

        path = str(GEN_DIR / f'{job_id}.mp4')
        from diffusers.utils import export_to_video
        export_to_video(frames, path, fps=7)
        jobs[job_id].update({'status': 'done', 'file': path, 'progress': 100, 'seed': actual})
        print(f'\n✅ SVD selesai → {path}')

    except Exception as e:
        print(f'\n❌ SVD error: {e}')
        jobs[job_id].update({'status': 'failed', 'error': str(e)[:300]})
        clear_memory(force=True)
    finally:
        model_lock = False


# ── CogVideoX — Image to Video HD ────────────────────────────────
def run_cog(job_id: str, req: dict):
    global gen_pipe, current_model_name, model_lock
    try:
        model_lock = True
        check_memory_safe()

        if current_model_name != 'cogvideox':
            clear_memory()
            print('\n🧠 Loading CogVideoX-5B I2V (~17GB, tunggu 3-5 menit)...')
            from diffusers import CogVideoXImageToVideoPipeline
            gen_pipe = CogVideoXImageToVideoPipeline.from_pretrained(
                'THUDM/CogVideoX-5b-I2V',
                torch_dtype=torch.bfloat16, low_cpu_mem_usage=True
            )
            gen_pipe.enable_sequential_cpu_offload()
            gen_pipe.unet.enable_forward_chunking(chunk_size=1) if hasattr(gen_pipe, 'unet') else None
            gen_pipe.vae.enable_tiling()
            gen_pipe.vae.enable_slicing()
            current_model_name = 'cogvideox'
            print(' ✓ CogVideoX Ready')

        img = Image.open(io.BytesIO(base64.b64decode(req.get('image_base64', '')))).convert('RGB')
        img = img.resize((req.get('width', 480), req.get('height', 272)))

        seed   = req.get('seed', -1)
        actual = seed if seed != -1 else int(np.random.randint(0, 1_000_000))
        gen    = torch.Generator(device='cpu').manual_seed(actual)
        n_frames = min(req.get('num_frames', 5), 5)

        with torch.inference_mode():
            frames = gen_pipe(
                prompt=req.get('prompt', 'smooth cinematic motion'),
                image=img,
                num_inference_steps=15, num_frames=n_frames,
                guidance_scale=6.0, generator=gen,
                callback_on_step_end=lambda pipe, i, t, kw: (_show_progress(job_id, i, 25, 'CogVideo'), kw)[1]
            ).frames[0]

        path = str(GEN_DIR / f'{job_id}.mp4')
        from diffusers.utils import export_to_video
        export_to_video(frames, path, fps=7)
        jobs[job_id].update({'status': 'done', 'file': path, 'progress': 100, 'seed': actual})
        print(f'\n✅ CogVideoX selesai → {path}')

    except torch.cuda.OutOfMemoryError:
        clear_memory(force=True)
        jobs[job_id].update({'status': 'failed', 'error': 'Out of memory — coba kurangi num_frames atau resolusi'})
    except Exception as e:
        print(f'\n❌ CogVideo error: {e}')
        jobs[job_id].update({'status': 'failed', 'error': str(e)[:300]})
        clear_memory(force=True)
    finally:
        model_lock = False


print('✅ AI Video Generator functions siap!')
print('   run_sd15  → Text to Image (SD 1.5)')
print('   run_svd   → Image to Video (SVD-XT)')
print('   run_cog   → Image to Video HD (CogVideoX-5B)')

✅ AI Video Generator functions siap!
   run_sd15  → Text to Image (SD 1.5)
   run_svd   → Image to Video (SVD-XT)
   run_cog   → Image to Video HD (CogVideoX-5B)


## 🌐 CELL 5 — VAX Model Unified FastAPI Server + ngrok

### Endpoint Lengkap:
| Method | Endpoint | Fungsi |
|---|---|---|
| POST | `/swap/image` | Face swap foto |
| POST | `/swap/video` | Face swap video |
| POST | `/generate/image` | Generate gambar dari teks |
| POST | `/generate/video` | Generate video dari gambar |
| GET | `/status/{id}` | Cek progress semua job |
| GET | `/result/{id}` | Download hasil semua job |
| GET | `/health` | Info server & VRAM |
| POST | `/clear_memory` | Bersihkan GPU memory |
| GET | `/docs` | Swagger UI (auto) |

In [ ]:
# ── HF MONKEYPATCH ──
import huggingface_hub
try:
    from huggingface_hub import cached_download
except ImportError:
    import huggingface_hub.file_download
    huggingface_hub.cached_download = huggingface_hub.hf_hub_download
    huggingface_hub.file_download.cached_download = huggingface_hub.hf_hub_download
try:
    from huggingface_hub.utils import HfFolder
except ImportError:
    try:
        from huggingface_hub import HfFolder
    except ImportError:
        class HfFolder:
            @classmethod
            def get_token(cls): import os; return os.environ.get('HF_TOKEN')
            @classmethod
            def save_token(cls, token): pass
            @classmethod
            def delete_token(cls): pass
    import huggingface_hub.utils
    huggingface_hub.utils.HfFolder = HfFolder
    huggingface_hub.HfFolder = HfFolder
import huggingface_hub.constants
if not hasattr(huggingface_hub.constants, 'HF_HUB_ENABLE_HF_TRANSFER'):
    huggingface_hub.constants.HF_HUB_ENABLE_HF_TRANSFER = False
# ───────────────────

import time, os, uuid, shutil, asyncio
import torch, nest_asyncio
from pathlib import Path

import uvicorn
from fastapi import FastAPI, File, UploadFile, Form, BackgroundTasks, HTTPException
from fastapi.responses import FileResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok, conf

nest_asyncio.apply()

# ══════════════════════════════════════════════════════════════════
#  ╔══════════════════════════════════════╗
#  ║  ISI NGROK TOKEN KAMU DI BAWAH INI  ║
#  ╚══════════════════════════════════════╝
#  Daftar gratis di: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = '3ChitPHrdVx5vCS2ObT3PiSmcKT_5VMi8Ms4gZcPA2cp34a4t'
# ══════════════════════════════════════════════════════════════════

WORK_DIR   = Path('/kaggle/working')
OUTPUT_DIR = WORK_DIR / 'output'
UPLOAD_DIR = WORK_DIR / 'uploads'
GEN_DIR    = WORK_DIR / 'generated'

for d in [OUTPUT_DIR, UPLOAD_DIR, GEN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── FastAPI App ───────────────────────────────────────────────────
app = FastAPI(
    title='VAX Model API',
    description='Unified AI Engine: Face Swap + AI Video Generator',
    version='1.0.0'
)
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'], allow_credentials=True,
    allow_methods=['*'], allow_headers=['*'],
)

# ── Pydantic Schemas ──────────────────────────────────────────────
class GenerateImageReq(BaseModel):
    prompt: str
    negative_prompt: str = 'blurry, low quality, distorted'
    width: int = 512
    height: int = 512
    num_inference_steps: int = 20
    guidance_scale: float = 7.5
    seed: int = -1

class GenerateVideoReq(BaseModel):
    image_base64: str
    prompt: str = ''
    model: str = 'svd'          # 'svd' atau 'cogvideox'
    num_frames: int = 14
    width: int = 480
    height: int = 272
    seed: int = -1


# ══════════════════════════════════════════════════════════════════
#  BACKGROUND TASKS — FACE SWAP
# ══════════════════════════════════════════════════════════════════

def _bg_image_swap(job_id, source_path, target_path, enhance):
    try:
        jobs[job_id]['status'] = 'processing'
        out = str(OUTPUT_DIR / f'{job_id}.jpg')
        process_image(source_path, target_path, out, enhance)  # dari Cell 3
        jobs[job_id].update({'status': 'done', 'file': out, 'progress': 100})
    except Exception as e:
        jobs[job_id].update({'status': 'failed', 'error': str(e)})

def _bg_video_swap(job_id, source_path, target_path, enhance):
    try:
        jobs[job_id]['status'] = 'processing'
        raw  = str(OUTPUT_DIR / f'{job_id}_raw.mp4')
        out  = str(OUTPUT_DIR / f'{job_id}.mp4')
        process_video(source_path, target_path, raw, enhance)  # dari Cell 3
        import subprocess
        r = subprocess.run([
            'ffmpeg', '-y', '-i', raw,
            '-c:v', 'libx264', '-profile:v', 'baseline', '-level', '3.0',
            '-pix_fmt', 'yuv420p', '-movflags', '+faststart',
            '-c:a', 'aac', '-b:a', '128k', '-avoid_negative_ts', 'make_zero',
            out
        ], capture_output=True, text=True, timeout=600)
        if r.returncode != 0:
            shutil.copy(raw, out)
        try: Path(raw).unlink(missing_ok=True)
        except: pass
        jobs[job_id].update({'status': 'done', 'file': out, 'progress': 100})
    except Exception as e:
        jobs[job_id].update({'status': 'failed', 'error': str(e)})


# ══════════════════════════════════════════════════════════════════
#  ENDPOINTS — FACE SWAP
# ══════════════════════════════════════════════════════════════════

@app.post('/swap/image', tags=['Face Swap'])
async def swap_image(
    background_tasks: BackgroundTasks,
    source: UploadFile = File(..., description='Foto muka baru (jpg/png)'),
    target: UploadFile = File(..., description='Foto/target yang diganti'),
    enhance: bool = Form(True, description='Enhance dengan GFPGAN')
):
    """Swap wajah pada gambar. Upload source (muka baru) dan target (foto target)."""
    job_id   = f'swap_{uuid.uuid4().hex[:8]}'
    src_path = str(UPLOAD_DIR / f'{job_id}_src{Path(source.filename).suffix}')
    tgt_path = str(UPLOAD_DIR / f'{job_id}_tgt{Path(target.filename).suffix}')
    with open(src_path, 'wb') as f: shutil.copyfileobj(source.file, f)
    with open(tgt_path, 'wb') as f: shutil.copyfileobj(target.file, f)
    jobs[job_id] = {'type': 'face_swap_image', 'status': 'queued',
                    'file': None, 'error': None, 'progress': 0, 'created': time.time()}
    background_tasks.add_task(_bg_image_swap, job_id, src_path, tgt_path, enhance)
    return {'job_id': job_id, 'status': 'queued',
            'message': f'Cek: GET /status/{job_id} | Download: GET /result/{job_id}'}


@app.post('/swap/video', tags=['Face Swap'])
async def swap_video(
    background_tasks: BackgroundTasks,
    source: UploadFile = File(..., description='Foto muka baru (jpg/png)'),
    target: UploadFile = File(..., description='Video target (mp4)'),
    enhance: bool = Form(False, description='Enhance tiap 3 frame (lambat untuk video panjang)')
):
    """Swap wajah pada video. Upload source face dan target video."""
    job_id   = f'swapv_{uuid.uuid4().hex[:8]}'
    src_path = str(UPLOAD_DIR / f'{job_id}_src{Path(source.filename).suffix}')
    tgt_path = str(UPLOAD_DIR / f'{job_id}_tgt{Path(target.filename).suffix}')
    with open(src_path, 'wb') as f: shutil.copyfileobj(source.file, f)
    with open(tgt_path, 'wb') as f: shutil.copyfileobj(target.file, f)
    jobs[job_id] = {'type': 'face_swap_video', 'status': 'queued',
                    'file': None, 'error': None, 'progress': 0, 'created': time.time()}
    background_tasks.add_task(_bg_video_swap, job_id, src_path, tgt_path, enhance)
    return {'job_id': job_id, 'status': 'queued',
            'message': f'Cek: GET /status/{job_id} | Download: GET /result/{job_id}'}


# ══════════════════════════════════════════════════════════════════
#  ENDPOINTS — AI GENERATION
# ══════════════════════════════════════════════════════════════════

@app.post('/generate/image', tags=['AI Generation'])
async def generate_image(req: GenerateImageReq, background_tasks: BackgroundTasks):
    """Generate gambar dari text prompt menggunakan Stable Diffusion v1.5."""
    job_id = f'img_{uuid.uuid4().hex[:8]}'
    jobs[job_id] = {'type': 'gen_image', 'status': 'processing',
                    'file': None, 'error': None, 'progress': 0, 'created': time.time()}
    background_tasks.add_task(run_sd15, job_id, req.model_dump())   # dari Cell 4
    return {'success': True, 'job_id': job_id,
            'message': f'Cek: GET /status/{job_id} | Download: GET /result/{job_id}'}


@app.post('/generate/video', tags=['AI Generation'])
async def generate_video(req: GenerateVideoReq, background_tasks: BackgroundTasks):
    """Animasikan gambar menjadi video. model: 'svd' (cepat) atau 'cogvideox' (HD, lambat)."""
    job_id = f'vid_{uuid.uuid4().hex[:8]}'
    jobs[job_id] = {'type': f'gen_video_{req.model}', 'status': 'processing',
                    'file': None, 'error': None, 'progress': 0, 'created': time.time()}
    fn = run_cog if req.model == 'cogvideox' else run_svd   # dari Cell 4
    background_tasks.add_task(fn, job_id, req.dict())
    return {'success': True, 'job_id': job_id,
            'message': f'Cek: GET /status/{job_id} | Download: GET /result/{job_id}'}


# ══════════════════════════════════════════════════════════════════
#  ENDPOINTS — COMMON
# ══════════════════════════════════════════════════════════════════

@app.get('/status/{job_id}', tags=['Common'])
async def get_status(job_id: str):
    """Cek status dan progress job (face swap atau generate)."""
    if job_id not in jobs:
        raise HTTPException(status_code=404, detail='Job tidak ditemukan')
    info = jobs[job_id]
    if time.time() - info.get('created', 0) > 3600:
        return {'job_id': job_id, 'status': 'expired'}
    res = {'job_id': job_id, 'type': info.get('type'),
           'status': info['status'], 'progress': info.get('progress', 0)}
    if info['status'] == 'done':
        res['download_url'] = f'/result/{job_id}'
    if info['status'] == 'failed':
        res['error'] = info.get('error')
    if 'seed' in info:
        res['seed'] = info['seed']
    return res


@app.get('/result/{job_id}', tags=['Common'])
async def get_result(job_id: str):
    """Download hasil job (gambar atau video)."""
    if job_id not in jobs:
        raise HTTPException(status_code=404, detail='Job tidak ditemukan')
    info = jobs[job_id]
    if info['status'] != 'done':
        raise HTTPException(status_code=400, detail=f'Job belum selesai: {info["status"]}')
    fp = info.get('file')
    if not fp or not os.path.exists(fp):
        raise HTTPException(status_code=404, detail='File tidak ditemukan di server')
    if fp.endswith('.mp4'):   mt = 'video/mp4'
    elif fp.endswith('.png'): mt = 'image/png'
    else:                     mt = 'image/jpeg'
    return FileResponse(fp, media_type=mt, filename=f'vax_{job_id}{Path(fp).suffix}')


@app.get('/health', tags=['Common'])
async def health():
    """Info status server, GPU, dan VRAM saat ini."""
    free, total = get_free_memory()                      # dari Cell 4
    active = len([j for j in jobs.values() if j['status'] == 'processing'])
    return {
        'status'        : 'online',
        'engine'        : 'VAX Model v1.0 (Face Swap + AI Video)',
        'gpu'           : torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'vram'          : f'{free:.1f}/{total:.1f} GB' if total > 0 else 'N/A',
        'model_loaded'  : current_model_name or 'none',  # dari Cell 4
        'active_jobs'   : active,
        'total_jobs'    : len(jobs),
    }


@app.post('/clear_memory', tags=['Common'])
async def clear_memory_api():
    """Paksa bersihkan GPU memory dan unload model yang sedang aktif."""
    clear_memory(force=True)                             # dari Cell 4
    free, _ = get_free_memory()
    return {'status': 'success', 'vram_free_gb': round(free, 2)}


# ══════════════════════════════════════════════════════════════════
#  PERIODIC CLEANUP
# ══════════════════════════════════════════════════════════════════

async def _periodic_cleanup():
    while True:
        await asyncio.sleep(300)   # setiap 5 menit
        now = time.time()
        to_del = [jid for jid, j in jobs.items() if now - j.get('created', 0) > 3600]
        for jid in to_del:
            fp = jobs[jid].get('file')
            if fp and os.path.exists(fp):
                try: os.remove(fp)
                except: pass
            del jobs[jid]
        if to_del:
            print(f'🧹 Removed {len(to_del)} expired jobs')


# ══════════════════════════════════════════════════════════════════
#  START NGROK + UVICORN
# ══════════════════════════════════════════════════════════════════
try:
    conf.get_default().auth_token = NGROK_TOKEN
    ngrok.kill()
    time.sleep(1)
    public_url = ngrok.connect(8000).public_url
except Exception as e:
    print(f'⚠️ Ngrok error: {e}')
    public_url = 'http://localhost:8000'

free, total = get_free_memory()
print('=' * 62)
print('🚀  VAX MODEL API — ONLINE')
print('=' * 62)
print(f'📡  Public URL : {public_url}')
print(f'📖  Swagger UI : {public_url}/docs')
print(f'💾  VRAM       : {free:.1f} / {total:.1f} GB Free')
print('─' * 62)
print('  FACE SWAP:')
print(f'    POST  {public_url}/swap/image   → swap foto')
print(f'    POST  {public_url}/swap/video   → swap video')
print('  AI GENERATE:')
print(f'    POST  {public_url}/generate/image → text to image')
print(f'    POST  {public_url}/generate/video → image to video')
print('  COMMON:')
print(f'    GET   {public_url}/status/{{id}}  → cek progress')
print(f'    GET   {public_url}/result/{{id}}  → download hasil')
print(f'    GET   {public_url}/health        → status server')
print('=' * 62)

async def _main():
    config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning')
    server = uvicorn.Server(config)
    asyncio.create_task(_periodic_cleanup())
    await server.serve()

await _main()

🚀  VAX MODEL API — ONLINE
📡  Public URL : https://seldom-glamorous-prodigal.ngrok-free.dev
📖  Swagger UI : https://seldom-glamorous-prodigal.ngrok-free.dev/docs
💾  VRAM       : 12.3 / 14.6 GB Free
──────────────────────────────────────────────────────────────
  FACE SWAP:
    POST  https://seldom-glamorous-prodigal.ngrok-free.dev/swap/image   → swap foto
    POST  https://seldom-glamorous-prodigal.ngrok-free.dev/swap/video   → swap video
  AI GENERATE:
    POST  https://seldom-glamorous-prodigal.ngrok-free.dev/generate/image → text to image
    POST  https://seldom-glamorous-prodigal.ngrok-free.dev/generate/video → image to video
  COMMON:
    GET   https://seldom-glamorous-prodigal.ngrok-free.dev/status/{id}  → cek progress
    GET   https://seldom-glamorous-prodigal.ngrok-free.dev/result/{id}  → download hasil
    GET   https://seldom-glamorous-prodigal.ngrok-free.dev/health        → status server


/tmp/ipykernel_58/2091705218.py:152: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  background_tasks.add_task(run_sd15, job_id, req.dict())   # dari Cell 4



🖼️ Loading Stable Diffusion v1.5...

❌ SD1.5 error: cannot import name 'cached_download' from 'huggingface_hub' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py)

🧹 Clearing GPU memory...


## 🧪 CELL 6 — Test Manual (Jalankan di Cell Terpisah setelah Server Aktif)

In [ ]:
# ─── Test Health Check ────────────────────────────────────────────
import requests

BASE_URL = public_url   # Otomatis dari Cell 5

resp = requests.get(f'{BASE_URL}/health')
print('📊 Health Check:')
import json; print(json.dumps(resp.json(), indent=2))

In [ ]:
# ─── Test Generate Image ──────────────────────────────────────────
import requests, json, time

BASE_URL = public_url

payload = {
    'prompt': 'a stunning Indonesian landscape, rice fields, dramatic sunset, cinematic, 8k',
    'negative_prompt': 'blurry, low quality, watermark',
    'width': 512, 'height': 512,
    'num_inference_steps': 20,
    'seed': 42
}
r = requests.post(f'{BASE_URL}/generate/image', json=payload)
data = r.json()
job_id = data['job_id']
print(f'Job ID: {job_id}')

# Poll sampai selesai
while True:
    status = requests.get(f'{BASE_URL}/status/{job_id}').json()
    print(f"  Status: {status['status']} | Progress: {status.get('progress', 0)}%")
    if status['status'] in ('done', 'failed', 'expired'):
        break
    time.sleep(3)

if status['status'] == 'done':
    img_data = requests.get(f'{BASE_URL}/result/{job_id}').content
    with open('/kaggle/working/test_output.png', 'wb') as f:
        f.write(img_data)
    print('✅ Gambar disimpan: /kaggle/working/test_output.png')
    from IPython.display import Image as IPImage, display
    display(IPImage('/kaggle/working/test_output.png'))

In [ ]:
# ─── Test Face Swap Image ─────────────────────────────────────────
# Ganti path di bawah dengan gambar yang ada di /kaggle/input
import requests, time

BASE_URL = public_url

SOURCE_IMG = '/kaggle/input/your_source_face.jpg'  # <-- ganti ini
TARGET_IMG = '/kaggle/input/your_target_photo.jpg' # <-- ganti ini

with open(SOURCE_IMG, 'rb') as s, open(TARGET_IMG, 'rb') as t:
    r = requests.post(
        f'{BASE_URL}/swap/image',
        files={'source': s, 'target': t},
        data={'enhance': True}
    )

data = r.json()
job_id = data['job_id']
print(f'Job ID: {job_id}')

while True:
    status = requests.get(f'{BASE_URL}/status/{job_id}').json()
    print(f"  {status['status']} | {status.get('progress', 0)}%")
    if status['status'] in ('done', 'failed', 'expired'):
        break
    time.sleep(2)

if status['status'] == 'done':
    result = requests.get(f'{BASE_URL}/result/{job_id}').content
    out_path = '/kaggle/working/faceswap_result.jpg'
    with open(out_path, 'wb') as f: f.write(result)
    print(f'✅ Hasil disimpan: {out_path}')
    from IPython.display import Image as IPImage, display
    display(IPImage(out_path))